In [1]:
import numpy as np 
import pandas as pd 

In [2]:
df = pd.read_csv("Train.csv")

In [3]:
df.head()

,ID,team_id,country,team_code,confederation_name,region_name,tournament_id,tournament_name,year,matches_played,total_goals,stage_reached
0,WC-1930_ARG,T-03,Argentina,ARG,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,5,18,final
1,WC-1930_BEL,T-06,Belgium,BEL,Union of European Football Associations,Europe,WC-1930,1930 FIFA Men's World Cup,1930,2,0,group stage
2,WC-1930_BOL,T-07,Bolivia,BOL,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,2,0,group stage
3,WC-1930_BRA,T-09,Brazil,BRA,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,2,5,group stage
4,WC-1930_CHL,T-13,Chile,CHL,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,3,5,group stage


In [4]:
df.shape

(489, 12)

In [6]:
df['country'].nunique()

85

In [7]:
df.columns

Index(['ID', 'team_id', 'country', 'team_code', 'confederation_name',
       'region_name', 'tournament_id', 'tournament_name', 'year',
       'matches_played', 'total_goals', 'stage_reached'],
      dtype='object')

In [8]:
from sklearn.metrics import root_mean_squared_error, f1_score, classification_report
from sklearn.preprocessing import OrdinalEncoder

In [18]:
import pandas as pd
import numpy as np

# 1. Load the training dataset
df = pd.read_csv("Train.csv")

# 2. Check the size of the data (rows, columns)
print("Dataset shape:", df.shape)

# 3. Look at the first 5 rows
df.head()

# 4. Check for missing values
print("\nMissing values in each column:")
df.isnull().sum()

Dataset shape: (489, 12)

Missing values in each column:


ID                    0
team_id               0
country               0
team_code             0
confederation_name    0
region_name           0
tournament_id         0
tournament_name       0
year                  0
matches_played        0
total_goals           0
stage_reached         0
dtype: int64

In [19]:
df.head()

,ID,team_id,country,team_code,confederation_name,region_name,tournament_id,tournament_name,year,matches_played,total_goals,stage_reached
0,WC-1930_ARG,T-03,Argentina,ARG,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,5,18,final
1,WC-1930_BEL,T-06,Belgium,BEL,Union of European Football Associations,Europe,WC-1930,1930 FIFA Men's World Cup,1930,2,0,group stage
2,WC-1930_BOL,T-07,Bolivia,BOL,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,2,0,group stage
3,WC-1930_BRA,T-09,Brazil,BRA,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,2,5,group stage
4,WC-1930_CHL,T-13,Chile,CHL,South American Football Confederation,South America,WC-1930,1930 FIFA Men's World Cup,1930,3,5,group stage


In [20]:
# Clean any accidental spaces and make lowercase
df['stage_reached'] = df['stage_reached'].str.strip().str.lower()

# Map stages from 0 (Group Stage) to 6 (Champion)
stage_map = {
    'group': 0,
    'roundof32': 1,
    'roundof16': 2,
    'qf': 3,
    'sf': 4,
    'runnerup': 5,
    'champion': 6
}
df['stage_number'] = df['stage_reached'].map(stage_map)

# Create a reverse map so we can turn numbers back into words later
number_to_stage = {v: k for k, v in stage_map.items()}

# Quick summary of goals and stages
print("\nAverage goals scored per stage:")
print(df.groupby('stage_reached')['total_goals'].mean().round(2))



Average goals scored per stage:
stage_reached
final                 13.48
final round           14.50
group stage            2.65
quarter-finals         7.18
round of 16            4.40
second group stage     7.25
semi-finals            7.00
third-place match     11.80
Name: total_goals, dtype: float64


In [21]:
# Sort by team and year so history is in the right time order
df = df.sort_values(['team_id', 'year']).reset_index(drop=True)

# Calculate goals per match for historical reference
df['goals_per_match'] = df['total_goals'] / df['matches_played']

# 1. How many World Cups had this team played BEFORE this year?
df['past_appearances'] = df.groupby('team_id').cumcount()

# 2. Average goals scored in PREVIOUS World Cups (.shift(1) ensures we don't peek at the current year!)
df['past_avg_goals'] = df.groupby('team_id')['total_goals'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 3. Average goals per match in PREVIOUS World Cups
df['past_avg_gpm'] = df.groupby('team_id')['goals_per_match'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 4. Average stage reached in PREVIOUS World Cups
df['past_avg_stage'] = df.groupby('team_id')['stage_number'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 5. Best stage ever reached before this year
df['past_best_stage'] = df.groupby('team_id')['stage_number'].transform(
    lambda x: x.shift(1).expanding().max()
)

# If a team is playing its VERY FIRST World Cup, its past stats will be NaN (empty).
# Let's fill those empty values with 0 (or the overall average) so the model doesn't crash.
df = df.fillna(0)

# Convert text columns (confederation & region) into numbers using One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['confederation_name', 'region_name'], drop_first=True)

print("Features created successfully! Total columns now:", df_encoded.shape[1])


Features created successfully! Total columns now: 32


In [22]:
# Pick the columns our model is allowed to use as clues (Features)
ignore_cols = [
    'ID', 'team_id', 'country', 'team_code', 'tournament_id', 
    'tournament_name', 'matches_played', 'total_goals', 
    'stage_reached', 'stage_number', 'goals_per_match'
]
features = [col for col in df_encoded.columns if col not in ignore_cols]

# Split by year: Train on past tournaments, Validate on 2022
train_data = df_encoded[df_encoded['year'] < 2022].copy()
val_data = df_encoded[df_encoded['year'] == 2022].copy()

X_train = train_data[features]
X_val = val_data[features]

print(f"Training rows (1930-2018): {len(X_train)}")
print(f"Validation rows (2022): {len(X_val)}")


Training rows (1930-2018): 457
Validation rows (2022): 32


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

y_train_goals = train_data['total_goals']
y_val_goals = val_data['total_goals']

# 1. Create and train the model
goal_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
goal_model.fit(X_train, y_train_goals)

# 2. Make predictions on 2022
pred_goals = goal_model.predict(X_val)

# 3. Check how close we were (RMSE)
rmse = root_mean_squared_error(y_val_goals, pred_goals)
print(f"Goals Prediction RMSE on 2022: {rmse:.2f} goals")


Goals Prediction RMSE on 2022: 3.50 goals


In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

y_train_stage = train_data['stage_number']
y_val_stage = val_data['stage_number']

# 1. Create and train the classifier
stage_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
stage_model.fit(X_train, y_train_stage)

# 2. Predict stage numbers for 2022
pred_stage_num = stage_model.predict(X_val)

# 3. Check F1 Score
f1 = f1_score(y_val_stage, pred_stage_num, average='weighted')
print(f"Stage Prediction F1 Score on 2022: {f1:.2f}")


Stage Prediction F1 Score on 2022: 1.00
